# Gradient boosting: residuals and split gain

Walk through additive squared-error boosting, then calculate XGBoost-style gradient statistics. The small numbers illustrate the objective, not a complete implementation of XGBoost, LightGBM, or CatBoost.

In [1]:
targets = [1.0, 1.0, 3.0, 3.0]
learning_rate = 0.5
pred = [sum(targets)/len(targets)] * len(targets)
def half_squared_error(ys, predictions):
    return sum(0.5*(y-p)**2 for y,p in zip(ys,predictions))/len(ys)
print('round 0:', pred, 'loss:', round(half_squared_error(targets,pred), 4))
for round_id in (1,2):
    residuals = [y-p for y,p in zip(targets,pred)]
    left_value = sum(residuals[:2])/2
    right_value = sum(residuals[2:])/2
    stump = [left_value]*2 + [right_value]*2
    pred = [p + learning_rate*correction for p,correction in zip(pred,stump)]
    print(f'round {round_id}: leaf values={left_value:.2f}, {right_value:.2f};'
          f' predictions={[round(p, 2) for p in pred]};'
          f' loss={half_squared_error(targets,pred):.4f}')

round 0: [2.0, 2.0, 2.0, 2.0] loss: 0.5
round 1: leaf values=-1.00, 1.00; predictions=[1.5, 1.5, 2.5, 2.5]; loss=0.1250
round 2: leaf values=-0.50, 0.50; predictions=[1.25, 1.25, 2.75, 2.75]; loss=0.0312


## Curvature-aware leaf scores

For loss $\tfrac12(y-\hat y)^2$, each per-row gradient is $g=\hat y-y$ and Hessian is $h=1$. The regularized leaf score is $-G/(H+\lambda)$, where $G$ and $H$ are sums within that leaf.

In [2]:
baseline = 2.0
G_left = sum(baseline-y for y in targets[:2])
G_right = sum(baseline-y for y in targets[2:])
H_left = H_right = 2.0
l2, gamma = 1.0, 0.1
leaf = lambda G,H: -G/(H+l2)
G_parent, H_parent = G_left+G_right, H_left+H_right
gain = 0.5*(G_left**2/(H_left+l2) + G_right**2/(H_right+l2)
            - G_parent**2/(H_parent+l2)) - gamma
print('G left/right:', G_left, G_right)
print('regularized leaf scores:', round(leaf(G_left,H_left), 4), round(leaf(G_right,H_right), 4))
print('split gain after gamma:', round(gain, 4))

G left/right: 2.0 -2.0
regularized leaf scores: -0.6667 0.6667
split gain after gamma: 1.2333


### Try it

Increase $\lambda$ and $\gamma$; observe shrinkage and split gain. For a real comparison of boosting libraries, fit on training data only, stop on validation data, and report a final untouched test metric.